# Credit Risk Walkthrough

This notebook is a guided companion to the production-style pipeline in `run_analysis.py`. It focuses on understanding the business problem, Probability of Default (PD), feature engineering, model comparison, calibration, thresholds, and risk bands.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
from sklearn.model_selection import train_test_split
from src.data import load_uci_data, basic_quality_report, TARGET
from src.features import add_features
from src.model import fit_models, assign_risk_band
from src.business import risk_band_summary

## 1. Load and inspect the data

The target is next-month default. We care about estimated probability, not only a hard 0/1 classification.

In [ ]:
df = load_uci_data()
basic_quality_report(df)

In [ ]:
df_model = add_features(df)
X = df_model.drop(columns=[TARGET])
y = df_model[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
results = fit_models(X_train, X_test, y_train, y_test)
metrics = pd.DataFrame({name: result.metrics for name, result in results.items()}).T
metrics.round(4)

## 2. Interpret metrics correctly

ROC-AUC measures ranking, Average Precision is useful under class imbalance, Recall measures default capture at a selected threshold, and Brier Score assesses probability error. Accuracy alone is not the main decision metric.

In [ ]:
best_name = metrics['roc_auc'].astype(float).idxmax()
pred = results[best_name].predictions.copy()
pred['risk_band'] = assign_risk_band(pred['pd'])
risk_band_summary(pred)

## 3. Interview takeaway

A useful credit-risk model must be discussed in terms of discrimination, calibration, operating thresholds, business trade-offs, and governance limitations—not only model accuracy.